# CS13 Cell-Type Point-Cloud Visualization

Render the reconstructed CS13 point cloud from fixed camera views and generate a matching categorical legend.

This curated notebook targets the current Dynamo-free Spateo API. Edit the path/configuration cells for a new system before execution.


In [ ]:
import numpy as np

import pyvista as pv

pv.global_theme.transparent_background = True

import spateo as st


In [ ]:
cpo1 = [
    (7235.672822135923, -9333.743725966886, 29537.530999873827),
    (5265.3827, 317.52565000000004, 1200.0),
    (-0.9910735612049205, -0.13108990476291582, 0.024261763123203748),
]


In [ ]:
cpo2 = [
    (-134.8569326485603, -22618.34076227472, 15863.43005107714),
    (4720.1671, 112.69174999999996, 1200.0),
    (-0.9734993446053986, 0.06684381746631253, -0.21870283518827516),
]


In [ ]:
cpo3 = [
    (-5006.789177572386, -25557.216094653202, -115.0783686672523),
    (4720.1671, 112.69174999999996, 1200.0),
    (-0.9334155956236307, 0.35598914126682585, -0.04480019136890938),
]


## Load and validate data


In [ ]:
cs13 = st.read_h5ad("/DATA/User/gaomohan/CS13_Project/cs13/CS13.final.h5ad")
cs13


## Construct the point-cloud model


In [ ]:
cs13_pc, plot_cmap = st.tdr.construct_pc(
    adata=cs13.copy(),
    spatial_key="spatial",
    groupby="celltype",
    key_added="tissue",
    colormap="rainbow",
)


In [ ]:
st.pl.three_d_plot(
    model=cs13_pc,
    key="tissue",
    model_style="points",
    show_axes=True,
    colormap=plot_cmap,
    jupyter="static",  # False
    window_size=(1024, 1024),
    cpo=cpo1,
    # filename = f"./figures/cs13.pdf"
)


In [ ]:
st.pl.three_d_plot(
    model=cs13_pc,
    key="tissue",
    model_style="points",
    show_axes=True,
    colormap=plot_cmap,
    jupyter="static",  # False
    window_size=(1024, 1024),
    cpo=cpo2,
    # filename = f"./figures/cs13.pdf"
)


In [ ]:
st.pl.three_d_plot(
    model=cs13_pc,
    key="tissue",
    model_style="points",
    show_axes=True,
    colormap=plot_cmap,
    jupyter="static",  # False
    window_size=(1024, 1024),
    cpo=cpo3,
    # filename = f"./figures/cs13.pdf"
)


In [ ]:
print(cs13_pc.point_data.keys())


In [ ]:
tissues = np.asarray(cs13_pc.point_data["tissue"])
rgba = np.asarray(cs13_pc.point_data["tissue_rgba"])

print(tissues.shape)
print(rgba.shape)
print(rgba.min(), rgba.max())


In [ ]:
labels = []
colors = []

for tissue in np.unique(tissues):
    idx = np.where(tissues == tissue)[0][0]

    color = rgba[idx]

    # Spateo normally stores colors on a 0-1 scale.
    # Convert 0-255 RGBA values to 0-1 when necessary.
    if np.max(color) > 1:
        color = color / 255.0

    labels.append(str(tissue))
    colors.append(color)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="None",
        markerfacecolor=color,
        markeredgecolor=color,
        markersize=7,
        label=label,
    )
    for label, color in zip(labels, colors)
]

fig, ax = plt.subplots(figsize=(13, 3))

ax.axis("off")

ax.legend(
    handles=handles,
    ncol=5,  # Five legend entries per row.
    loc="center",
    frameon=False,
    fontsize=11,
    columnspacing=1.5,
    handletextpad=0.4,
    handlelength=0.8,
    borderaxespad=0,
)

plt.tight_layout()
plt.show()
